# 02a Ophyd trigger and read T8 asynchronous

This example demonstrates how to use the LabJack T8 with the ophyd library in an asynchronous manner.

## Summary:
1. We define a custom Ophyd Device for the LabJack T8, which includes methods for triggering and reading data.
2. We implement an asynchronous trigger using Python threading, allowing the main thread to remain responsive while waiting for the acquisition to complete.
3. The script handles both real hardware connections via LJM and simulated data, making it versatile for testing and development.
4. A manual execution loop triggers the device and reads back the averaged results.
5. This standalone test is useful for verifying hardware communication and Ophyd signal logic before integrating the device into a full Bluesky RunEngine scan.




# Preamble

## Preamble for Jupyter

In [ ]:
if 0:  # if True, display full output in Jupyter, not only last result
    from IPython.core.interactiveshell import InteractiveShell

    InteractiveShell.ast_node_interactivity = "all"  # type: ignore

from IPython.display import (
    display,  # noqa: F401
    Math,  # noqa: F401
)  # to display results in Jupyter, and to print in Latex format (not only for sympy)

## Example display Latex
# display(Math('\\pi =' + f'{np.pi:.6f}'))
# display(Math('\\sin(\\theta)'))

%load_ext autoreload
%autoreload 2


## Preamble Main Libraries and a few usefull constants

In [ ]:
import sys  # noqa: F401
import os  # noqa: F401

import numpy as np
import time
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go

# sys.path.append(os.environ['USERPROFILE'] + '\\workspace\\python\\libs\\wgTools')
# import wgutils as wgu
from scipy import constants

deg2rad = np.deg2rad(1)
rad2deg = np.rad2deg(1)
eps = np.finfo(np.float64).eps
hc = constants.value("inverse meter-electron volt relationship")

In [ ]:
# --- THE DETECTOR CLASS ---
import labjack_t8_ophyd as ljt8o

## Preamble for Matplotlib


In [ ]:
# # interative inline figures for JupyterLab need to be set by IPython magic commands
# # to check all the matplotlib backends, run
# %matplotlib --list
# # for vscode you want to use widget
# # For interactive figures in VS Code, use the 'widget' backend
# %matplotlib widget
# # if you export this code to a .py file, you need to run the line below for iterative ploting in ipython
# # get_ipython().run_line_magic('matplotlib', '--list')
# # interative inline figures for JupyterLab
# # get_ipython().run_line_magic('matplotlib', 'qt5')
# # for colab you may need to install ipympl
# # !pip install ipympl
# # Then run the magic
# # %matplotlib ipympl

# import matplotlib.pyplot as plt

# plt.style.use("default")

# params = {
#     "font.size": 10,
#     "legend.fontsize": "small",
#     "font.family": "serif",
#     "figure.facecolor": "white",
#     "axes.grid": True,
#     "figure.autolayout": True,
#     "axes.grid.axis": "both",
#     "mathtext.fontset": "stix",
#     "figure.dpi": 100,
#     "savefig.dpi": 300,
#     "figure.figsize": (6, 5),
#     "image.origin": "lower",
#     "image.interpolation": "none",
#     "image.cmap": "magma",
#     "lines.marker": "o",
#     "lines.linestyle": "-",
# }

# plt.rcParams.update(params)

## Preamble Plotly

In [ ]:
import plotly.colors

colors_plotly = plotly.colors.DEFAULT_PLOTLY_COLORS


def custom_layout(xlabel=None, ylabel=None, title=None, template="seaborn"):
    return go.Layout(
        template=template,  # Set the template
        width=800,  # Set the figure width
        height=600,  # Set the figure height
        font=dict(family="Georgia", size=14, color="#3B3B3B"),  # General font settings
        title=dict(
            text=title,
            font=dict(size=24),
            x=0.5,
            xanchor="center",
        ),
        xaxis_title=dict(text=xlabel, font=dict(size=18)),
        yaxis_title=dict(text=ylabel, font=dict(size=18)),
        xaxis=dict(tickfont=dict(size=16)),  # Set X-axis tick font size
        yaxis=dict(tickfont=dict(size=16)),  # Set Y-axis tick font size
    )


# initialize figures with something like
# plotly_fig = go.Figure(layout=custom_layout("Frequency [Hz]", "Amplitude [Volts]", fname))


## Local Functions

In [ ]:
def datenow_str():
    return datetime.now().isoformat(sep=" ", timespec="milliseconds")


# Setup Script Flags

In [ ]:
save_plot = False
close_connections_at_end = False

# Setup Hardware

## List Labjack Devices

In [ ]:
devices = ljt8o.detect_labjacks(verbose=True)

### Select a serial number

In [ ]:
_identifier = devices[0]["serial number"]
_identifier

## Opening connection to device with Serial number

NOTE that acquisition parameters are set here.


In [ ]:
print(f"[INFO] {datenow_str()}: Starting Detector Test with Ophyd...")
det = ljt8o.LabJackT8(
    name="test_lj",
    identifier=_identifier,
    active_AI_channels=[0, 1, 3, 5],
    acq_time=0.10,
    sample_rate=1000.0,  # max is 400kHz
    ranges={0: 10, "AIN1": 3.44},
    enable_waveforms=False,
    verbose=True,
)
print(f"[INFO] {datenow_str()}: ophyd LabJackT8 instance created with active channels: ", det.channel_names)

In [ ]:
det.acq_time

In [ ]:
det.AI_channels_name


In [ ]:
det.AI_channels_n

In [ ]:
det.AI_ranges

In [ ]:
det.AI_actual_range

## Range for each channel.

This is important to ensure you get the best resolution for your expected signal levels.

AIN range options for T8 (from LabJack documentation):

Ranges are: 

±11 V, ±9.6 V, ±4.8 V, ±2.4 V, ±1.2 V,

±0.6 V, ±0.3 V, ±0.15 V,

±0.075 V, ±0.036 V, ±0.018 V

In [ ]:
det.set_AI_range(0, 0.15)

In [ ]:
det.AI_ranges

In [ ]:
det.AI_actual_range

# Run stream

Note that this start and stop the stream. If we dont stop, data will continuouly be saved to the buffer.

In [ ]:
readings_list = []

for _i in range(2):
    print(f"[INFO] {datenow_str()} Triggering the detector...")
    _start_time = time.time()
    status = det.trigger()
    print(f"[INFO] {datenow_str()}: Detector triggered, waiting for acquisition to complete...")

    # Wait for the trigger to finish
    count = 0
    _start_time = time.time()

    while not status.done:
        # print(f"[INFO] {datenow_str()}: Waiting for detector to complete acquisition...")
        print(".", end="", flush=True)
        time.sleep(0.01)
        if time.time() - _start_time > det.acq_time * 5:  # Safety timeout
            print("[TIMEOUT] Status never marked as done!")
            break

    print("\nDONE! Time elapsed: {:.3f} s".format(time.time() - _start_time))
    print(f"[INFO] {datenow_str()}: Acquisition COMPLETED!. Actual scan rate: {det.last_scan_actual_rate:.3f} Hz")
    # # get the readings
    readings = det.read()
    readings_list.append(readings)

    # for _key in readings.keys():
    #     if "waveform" in _key or "raw" in _key:
    #         print(f"[RESULT] {datenow_str()}: {_key} SHAPE: {readings[_key]['value'].shape}")
    #     else:
    #         print(f"[RESULT] {datenow_str()}: {_key} VALUE: {readings[_key]['value']}")

In [ ]:
readings_list[0]


## Convert to pandas

In [ ]:
df = pd.DataFrame(readings_list[0]["test_lj_raw_block"]["value"], columns=["Time"] + det.channel_names)

for _i in range(1, len(readings_list)):
    print(_i)
    df = pd.concat(
        [df, pd.DataFrame(readings_list[_i]["test_lj_raw_block"]["value"], columns=["Time"] + det.channel_names)],
        ignore_index=True,
    )

df["Time"] -= df["Time"].min()

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

# Post Processing

In [ ]:
print("\n[POSTPROCESSING] Post Processing STARTED")


print("\n[POSTPROCESSING] Plotting data with Plotly...")
fig = go.Figure()
times = df["Time"]
for i in range(1, df.shape[1]):
    fig.add_trace(go.Scatter(x=times, y=df.iloc[:, i], mode="lines+markers", name=f"{df.columns[i]}"))

fig.update_layout(
    title="LabJack Data Acquisition",
    xaxis_title="Time (s)",
    yaxis_title="Voltage (V)",
    hovermode="x unified",
    template="plotly",
)
# Save HTML and open in VS Code webview
html_file = f"Results\\{datenow_str()}_labjack_acquisition_plot.html"

if save_plot:
    fig.write_html(html_file)
    print(f"[POSTPROCESSING] Plot saved to .\\{html_file}\n")

fig.show()

# END - Close connections

In [ ]:
if close_connections_at_end:
    det.close()
    print("[INFO] Detector connection closed.")